# Category Life List

This notebook generates and exports a life list for all sightings of the species in a category. For each species in that category, the total number of sightings of the species and the number of locations where sightings were recorded are shown alongside the species name. To use it, update the category in the first code cell, below, before running the notebook.

In [ ]:
# Species category to report on
category = "Birds"

In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb

In [ ]:
import sqlparse

# Construct the query
query = construct_query("category_life_list.sql", {
    "CATEGORY": category
})

# Show a pretty-printed form of the query
print(sqlparse.format(query, reindent=True, keyword_case='upper'))

In [ ]:
# Connect to the database and execute the query and read the results into a dataframe
df = query_data(query)

# Check there is some data
if not df.shape[0]:
    message = f"No data found for category '{category}'"
    raise ValueError(message)

In [ ]:
import pandas as pd

# Aggregate the data to produce the life list
life_list = (
    df
    .groupby("Species")
    .agg(
        Sightings=("Count", "sum"),
        Locations=("Location", pd.Series.nunique)
    )
    .reset_index()
    .sort_values(by="Species", ascending=True)
)


# Print the life list
with pd.option_context('display.max_rows', None,
                       'display.max_columns', None,
                       'display.precision', 3,
                       ):
    display(life_list)

In [ ]:
# Create the folder to hold exported reports
export_folder_path = get_export_folder_path()

# Export the life list
clean_category = clean_string(category)
export_file_name = f"{clean_category}-Life-List.xlsx"
export_to_spreadsheet(export_folder_path, f"{export_file_name}.xlsx", {
    "Life List": life_list
})
# life_list.to_excel(export_file_path.absolute(), sheet_name="Life List", index=False)